In [1]:
"""
GPU Post-Training Quantization (PTQ) — ResNet-50 / CIFAR-10
=============================================================
GPU-native PTQ benchmarking three GPU-capable approaches:

  1. ONNX Runtime  — FP32 → ONNX → INT8 (QDQ) via ORT quantization API
                     3 calibration methods: MinMax, Entropy, Percentile
                     Inference via CUDAExecutionProvider
  2. TensorRT      — FP32 → ONNX → TRT INT8 engine with calibration
                     IInt8EntropyCalibrator2; layer fusion (Conv+BN+ReLU)
                     (requires tensorrt + pycuda)
  3. PyTorch CUDA  — (a) Dynamic PTQ: Linear → qint8 on CUDA tensors
                     (b) PT2E: torch.export + X86InductorQuantizer → torch.compile

Mathematical foundation:
  Q(x)  = clip(round(x / S) + Z, q_min, q_max)   [quantization]
  x̂     = S · (Q(x) − Z)                          [dequantization]
  ε     = x − x̂,  |ε| ≤ S/2                      [bounded error]
  S     = (x_max − x_min) / (2^b − 1)             [scale from calibration]
  Z     = clip(round(−x_min / S), q_min, q_max)   [zero-point]

FLOPs note:
  FLOPs (op count) are IDENTICAL for FP32 and INT8.
  Quantization changes the storage dtype, NOT the number of operations.
  The GPU speedup is a hardware property: INT8 Tensor Core lanes process
  ~4× more elements per cycle than FP32 (Ampere), ~8× (Hopper).

Output:
  ptq_results.json          — per-method results in canonical schema
  __1__saved_models_PTQ/    — all serialised models
"""

# ── Imports ────────────────────────────────────────────────────────────────────
import os
import sys
import json
import time
import copy
import tempfile
import warnings
import argparse
import subprocess
from typing import Dict, List, Optional

import numpy as np

warnings.filterwarnings("ignore")

# ── Config ─────────────────────────────────────────────────────────────────────
BASELINE_MODEL_PATH   = "../__2__baseline_resnet50_cifar10.pth"
BASELINE_METRICS_PATH = "../__2__baseline_metrics.json"
OUTPUT_JSON           = "__1__PTQ_GPU.json"
SAVE_DIR              = "__1__saved_models_PTQ_GPU"

BATCH_SIZE     = 128
NUM_WORKERS    = 4
NUM_CLASSES    = 10
CALIB_SIZE     = 1024
CALIB_BATCHES  = 8
INFERENCE_RUNS = 50
INPUT_SHAPE    = (1, 3, 32, 32)

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

# ── Lazy dependency imports ────────────────────────────────────────────────────
def _try_import(name: str):
    try:
        import importlib
        return importlib.import_module(name)
    except ImportError:
        return None

torch        = _try_import("torch")
torchvision  = _try_import("torchvision")
ort          = _try_import("onnxruntime")
onnx         = _try_import("onnx")
fvcore       = _try_import("fvcore")
tensorrt     = _try_import("tensorrt")
pycuda       = _try_import("pycuda")


def install_deps():
    cmds = [
        "pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121",
        "pip install onnx onnxruntime-gpu",
        "pip install fvcore",
        "pip install tensorrt pycuda",
    ]
    for cmd in cmds:
        print(f"  $ {cmd}")
        subprocess.run(cmd.split(), check=False)


# ══════════════════════════════════════════════════════════════════════════════
# Model & Data
# ══════════════════════════════════════════════════════════════════════════════
def build_model(num_classes: int = 10):
    import torch.nn as nn
    from torchvision import models
    model = models.resnet50(weights=None)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, num_classes)
    return model

def load_baseline(path: str):
    model = build_model(NUM_CLASSES)
    model.load_state_dict(torch.load(path, map_location="cpu", weights_only=True))
    model.eval()
    return model

def get_test_loader():
    from torchvision import datasets, transforms
    from torch.utils.data import DataLoader
    tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = datasets.CIFAR10(root="../data", train=False, download=True, transform=tf)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True)

def get_calib_loader():
    from torchvision import datasets, transforms
    from torch.utils.data import DataLoader, Subset
    tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds  = datasets.CIFAR10(root="../data", train=True, download=True, transform=tf)
    sub = Subset(ds, list(range(CALIB_SIZE)))
    return DataLoader(sub, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True)


# ══════════════════════════════════════════════════════════════════════════════
# FLOPs & Parameter Counting
# ══════════════════════════════════════════════════════════════════════════════
def count_flops(model) -> Dict:
    """
    Compute FLOPs using fvcore (preferred) or manual hook-based fallback.

    FLOPs = 2 × MACs
      Conv2d  : MACs = (Cin/groups) × Cout × Kh × Kw × Hout × Wout
      Linear  : MACs = in_features × out_features

    FLOPs are IDENTICAL for FP32 and INT8 models.
    Quantization changes storage dtype, not arithmetic operation count.
    """
    if fvcore is not None:
        try:
            from fvcore.nn import FlopCountAnalysis
            x     = torch.randn(*INPUT_SHAPE)
            flops = FlopCountAnalysis(model.cpu().eval(), x)
            flops.unsupported_ops_warnings(False)
            total_macs   = flops.total()
            params_total = sum(p.numel() for p in model.parameters())
            return {
                "flops_G"     : round(total_macs * 2 / 1e9, 9),
                "params_total": int(params_total),
                "params_M"    : round(params_total / 1e6, 3),
                "tool"        : "fvcore",
            }
        except Exception:
            pass

    # Manual fallback
    import torch.nn as nn
    spatial  = {}
    handles  = []

    def make_hook(name):
        def hook(_, __, out):
            if hasattr(out, "shape"):
                spatial[name] = tuple(out.shape)
        return hook

    for name, mod in model.named_modules():
        handles.append(mod.register_forward_hook(make_hook(name)))
    x = torch.randn(*INPUT_SHAPE)
    with torch.no_grad():
        model.cpu().eval()(x)
    for h in handles:
        h.remove()

    total_flops = 0
    for name, mod in model.named_modules():
        if isinstance(mod, nn.Conv2d):
            shape = spatial.get(name)
            if shape is None:
                continue
            _, cout, hout, wout = shape
            cin = mod.in_channels
            kh  = mod.kernel_size[0] if isinstance(mod.kernel_size, tuple) else mod.kernel_size
            kw  = mod.kernel_size[1] if isinstance(mod.kernel_size, tuple) else mod.kernel_size
            total_flops += 2 * cout * (cin // mod.groups) * kh * kw * hout * wout
        elif isinstance(mod, nn.Linear):
            total_flops += 2 * mod.in_features * mod.out_features

    params_total = sum(p.numel() for p in model.parameters())
    return {
        "flops_G"     : round(total_flops / 1e9, 9),
        "params_total": int(params_total),
        "params_M"    : round(params_total / 1e6, 3),
        "tool"        : "manual",
    }

def count_params(model) -> Dict:
    """Count total and non-zero parameters."""
    total    = sum(p.numel() for p in model.parameters())
    nonzero  = sum(int((p != 0).sum().item()) for p in model.parameters())
    return {"params_total": int(total), "params_nonzero": int(nonzero)}


# ══════════════════════════════════════════════════════════════════════════════
# Disk Size
# ══════════════════════════════════════════════════════════════════════════════
def disk_size_mb(obj) -> float:
    """Return on-disk size in MB. Pass a file path (str) or a PyTorch model."""
    if isinstance(obj, str):
        return os.path.getsize(obj) / 1024 ** 2
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(obj, tmp)
        return os.path.getsize(tmp) / 1024 ** 2
    finally:
        if os.path.exists(tmp):
            os.remove(tmp)


# ══════════════════════════════════════════════════════════════════════════════
# Inference Timing
# ══════════════════════════════════════════════════════════════════════════════
def measure_gpu_throughput(model, device, n: int = INFERENCE_RUNS) -> Dict:
    """
    Timing via CUDA events — more accurate than wall-clock for GPU kernels.
    Returns the canonical inference_ms block.
    """
    model = model.to(device).eval()
    start_ev = torch.cuda.Event(enable_timing=True)
    end_ev   = torch.cuda.Event(enable_timing=True)

    dummy_batch  = torch.randn(BATCH_SIZE, 3, 32, 32, device=device)
    dummy_single = torch.randn(1,          3, 32, 32, device=device)

    # Warmup
    with torch.no_grad():
        for _ in range(10):
            model(dummy_batch)
    torch.cuda.synchronize(device)

    # Batch timing
    batch_timings = []
    with torch.no_grad():
        for _ in range(n):
            start_ev.record()
            model(dummy_batch)
            end_ev.record()
            torch.cuda.synchronize()
            batch_timings.append(start_ev.elapsed_time(end_ev))

    # Single-image timing
    with torch.no_grad():
        for _ in range(10):
            model(dummy_single)
    torch.cuda.synchronize(device)

    single_timings = []
    with torch.no_grad():
        for _ in range(n):
            start_ev.record()
            model(dummy_single)
            end_ev.record()
            torch.cuda.synchronize()
            single_timings.append(start_ev.elapsed_time(end_ev))

    batch_ms  = float(np.mean(batch_timings))
    single_ms = float(np.mean(single_timings))
    return {
        "single_img_gpu_ms"  : round(single_ms, 4),
        "batch128_gpu_ms"    : round(batch_ms, 4),
        "per_img_gpu_ms"     : round(batch_ms / BATCH_SIZE, 4),
        "throughput_imgs_sec": round(BATCH_SIZE / (batch_ms / 1000), 1),
        "timing_method"      : "CUDA events + torch.cuda.synchronize()",
    }

def measure_ort_throughput(session, n: int = INFERENCE_RUNS) -> Dict:
    """Wall-clock timing for ONNX Runtime sessions (no CUDA event API)."""
    input_name   = session.get_inputs()[0].name
    dummy_batch  = np.random.randn(BATCH_SIZE, 3, 32, 32).astype(np.float32)
    dummy_single = np.random.randn(1,          3, 32, 32).astype(np.float32)

    # Warmup
    for _ in range(5):
        session.run(None, {input_name: dummy_batch})

    # Batch
    t0 = time.perf_counter()
    for _ in range(n):
        session.run(None, {input_name: dummy_batch})
    batch_ms = (time.perf_counter() - t0) / n * 1000

    # Single
    for _ in range(5):
        session.run(None, {input_name: dummy_single})
    t0 = time.perf_counter()
    for _ in range(n):
        session.run(None, {input_name: dummy_single})
    single_ms = (time.perf_counter() - t0) / n * 1000

    return {
        "single_img_gpu_ms"  : round(single_ms, 4),
        "batch128_gpu_ms"    : round(batch_ms, 4),
        "per_img_gpu_ms"     : round(batch_ms / BATCH_SIZE, 4),
        "throughput_imgs_sec": round(BATCH_SIZE / (batch_ms / 1000), 1),
        "timing_method"      : "wall-clock (time.perf_counter) — ORT session",
    }


# ══════════════════════════════════════════════════════════════════════════════
# Canonical Result Entry Builder
# ══════════════════════════════════════════════════════════════════════════════
def build_entry(metrics: Dict, params: Dict,
                size_mb: Optional[float], flops_G: float,
                inference_ms: Dict) -> Dict:
    """
    Build a result entry that exactly matches the canonical output schema:

      {
        "accuracy":     float,
        "precision":    float,
        "recall":       float,
        "f1":           float,
        "params":       int,          # total parameter count
        "params_nonzero": int,        # non-zero parameter count
        "size_mb":      float | null,
        "flops_G":      float,        # identical for FP32 and INT8
        "inference_ms": {
          "single_img_gpu_ms":   float,
          "batch128_gpu_ms":     float,
          "per_img_gpu_ms":      float,
          "throughput_imgs_sec": float,
          "timing_method":       str,
        }
      }
    """
    return {
        "accuracy"      : round(metrics["accuracy"],  6),
        "precision"     : round(metrics["precision"], 6),
        "recall"        : round(metrics["recall"],    6),
        "f1"            : round(metrics["f1"],        6),
        "params"        : int(params["params_total"]),
        "params_nonzero": int(params["params_nonzero"]),
        "size_mb"       : round(size_mb, 6) if size_mb is not None else None,
        "flops_G"       : round(flops_G, 9),
        "inference_ms"  : inference_ms,
    }


# ══════════════════════════════════════════════════════════════════════════════
# Evaluation
# ══════════════════════════════════════════════════════════════════════════════
@torch.no_grad()
def evaluate_torch(model, loader, device) -> Dict:
    from sklearn.metrics import (accuracy_score, precision_score,
                                 recall_score, f1_score)
    model = model.to(device).eval()
    preds, labels = [], []
    for inputs, lbls in loader:
        preds.extend(model(inputs.to(device)).argmax(1).cpu().numpy())
        labels.extend(lbls.numpy())
    y_pred, y_true = np.array(preds), np.array(labels)
    return {
        "accuracy" : float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }

def evaluate_onnx(session, loader) -> Dict:
    from sklearn.metrics import (accuracy_score, precision_score,
                                 recall_score, f1_score)
    input_name = session.get_inputs()[0].name
    preds, labels = [], []
    for inputs, lbls in loader:
        out = session.run(None, {input_name: inputs.numpy()})[0]
        preds.extend(np.argmax(out, axis=1))
        labels.extend(lbls.numpy())
    y_pred, y_true = np.array(preds), np.array(labels)
    return {
        "accuracy" : float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }


# ══════════════════════════════════════════════════════════════════════════════
# ONNX Export
# ══════════════════════════════════════════════════════════════════════════════
def export_to_onnx(model, output_path: str, opset: int = 17) -> str:
    """
    Export FP32 PyTorch model to ONNX (opset 17+ required for QDQ INT8 ops).
    Dynamic axes enable variable batch size at inference.
    """
    dummy = torch.randn(*INPUT_SHAPE)
    torch.onnx.export(
        model.cpu().eval(), dummy, output_path,
        opset_version       = opset,
        input_names         = ["input"],
        output_names        = ["output"],
        dynamic_axes        = {"input": {0: "batch_size"}, "output": {0: "batch_size"}},
        do_constant_folding = True,
        export_params       = True,
    )
    print(f"        ✓ ONNX exported → {output_path}  "
          f"({os.path.getsize(output_path)/1024**2:.2f} MB, opset={opset})")
    return output_path


# ══════════════════════════════════════════════════════════════════════════════
# 1. ONNX Runtime — INT8 Static PTQ  (3 calibration methods)
# ══════════════════════════════════════════════════════════════════════════════
def run_onnx_ptq(fp32_model, test_loader, calib_loader,
                 flops_G: float, baseline_params: Dict) -> Dict:
    """
    Static INT8 quantization via onnxruntime-gpu.

    QDQ format:  QuantizeLinear / DequantizeLinear node pairs wrap each op.
    Weights:     INT8 per-channel symmetric
    Activations: UINT8 per-tensor affine (calibrated scale/zero-point)
    Inference:   CUDAExecutionProvider → cuDNN INT8 (Conv), cuBLASLt (MatMul)

    Calibration methods
    -------------------
    MinMax     : S = (x_max − x_min) / 255   [widest dynamic range]
    Entropy    : KL-divergence threshold search over activation histogram
    Percentile : Clips outliers at a fixed percentile before computing S/Z
    """
    import onnxruntime
    from onnxruntime.quantization import (
        quantize_static, CalibrationDataReader,
        QuantType, QuantFormat, CalibrationMethod,
    )

    print("\n  [1/3] ONNX Runtime — Static INT8 PTQ (QDQ)")
    os.makedirs(SAVE_DIR, exist_ok=True)

    fp32_onnx = "resnet50_fp32_tmp.onnx"
    export_to_onnx(fp32_model, fp32_onnx)

    class CIFARCalibReader(CalibrationDataReader):
        def __init__(self, loader, max_batches: int = CALIB_BATCHES):
            self._iter    = iter(loader)
            self._batches = 0
            self._max     = max_batches

        def get_next(self):
            if self._batches >= self._max:
                return None
            try:
                imgs, _ = next(self._iter)
                self._batches += 1
                return {"input": np.ascontiguousarray(imgs.numpy(), dtype=np.float32)}
            except StopIteration:
                return None

    calib_cfg = {
        "ort_minmax"     : CalibrationMethod.MinMax,
        "ort_entropy"    : CalibrationMethod.Entropy,
        "ort_percentile" : CalibrationMethod.Percentile,
    }

    results = {}
    providers = (
        ["CUDAExecutionProvider", "CPUExecutionProvider"]
        if "CUDAExecutionProvider" in onnxruntime.get_available_providers()
        else ["CPUExecutionProvider"]
    )
    sess_opts = onnxruntime.SessionOptions()
    sess_opts.graph_optimization_level = onnxruntime.GraphOptimizationLevel.ORT_ENABLE_ALL

    for key, calib_method in calib_cfg.items():
        label     = key.replace("ort_", "").capitalize()
        int8_path = f"resnet50_int8_{label.lower()}_tmp.onnx"
        save_path = os.path.join(SAVE_DIR, f"onnx_int8_{label.lower()}.onnx")
        print(f"        Calibration: {label:12s}", end="", flush=True)

        try:
            quantize_static(
                model_input             = fp32_onnx,
                model_output            = int8_path,
                calibration_data_reader = CIFARCalibReader(calib_loader),
                quant_format            = QuantFormat.QDQ,
                per_channel             = True,
                reduce_range            = False,
                weight_type             = QuantType.QInt8,
                activation_type         = QuantType.QUInt8,
                calibrate_method        = calib_method,
                extra_options           = {
                    "EnableSubgraph"                 : True,
                    "MatMulConstBOnly"               : False,
                    "AddQDQPairToWeight"             : True,
                    "QuantizeAllFixedZeroPointTensors": True,
                },
            )

            session  = onnxruntime.InferenceSession(int8_path, sess_opts,
                                                     providers=providers)
            metrics  = evaluate_onnx(session, test_loader)
            size_mb  = disk_size_mb(int8_path)
            infer_ms = measure_ort_throughput(session)

            # ONNX model has the same architecture → same param count as FP32
            results[key] = build_entry(
                metrics      = metrics,
                params       = baseline_params,
                size_mb      = size_mb,
                flops_G      = flops_G,
                inference_ms = infer_ms,
            )

            # Persist the quantised model
            import shutil
            shutil.copy(int8_path, save_path)

            print(f" → Acc: {metrics['accuracy']:.4f}  "
                  f"Disk: {size_mb:.2f} MB  "
                  f"Lat: {infer_ms['batch128_gpu_ms']:.1f} ms  "
                  f"({session.get_providers()[0]})")
            print(f"        ✓ Saved → {save_path}")

        except Exception as exc:
            print(f" → FAILED: {exc}")
            results[key] = None
        finally:
            if os.path.exists(int8_path):
                os.remove(int8_path)

    if os.path.exists(fp32_onnx):
        os.remove(fp32_onnx)

    return results


# ══════════════════════════════════════════════════════════════════════════════
# 2. TensorRT — INT8 PTQ with IInt8EntropyCalibrator2
# ══════════════════════════════════════════════════════════════════════════════
def run_tensorrt_ptq(fp32_model, test_loader, calib_loader,
                     flops_G: float, baseline_params: Dict) -> Optional[Dict]:
    """
    TensorRT INT8 engine built from ONNX via NVIDIA's builder API.

    Calibration: IInt8EntropyCalibrator2
      - Collects per-tensor activation histogram over calib batches
      - Finds threshold T* = argmin KL(original_hist || quantized_hist)
      - Sets S = T* / 127

    Additional optimisations applied at build time:
      - Conv + BN + ReLU kernel fusion → single GPU kernel
      - Memory layout optimisation (NHWC)
      - Kernel autotuning (engine is GPU-architecture-specific, NOT portable)
      - FP16 fallback for ops not suited to INT8
    """
    print("\n  [2/3] TensorRT — INT8 PTQ")
    try:
        import tensorrt as trt
        import pycuda.driver as cuda
        import pycuda.autoinit  # noqa: F401
    except ImportError as exc:
        print(f"        ⚠ TensorRT/pycuda not available: {exc}")
        print("          Install: pip install tensorrt pycuda")
        return None

    os.makedirs(SAVE_DIR, exist_ok=True)
    TRT_LOGGER = trt.Logger(trt.Logger.WARNING)
    fp32_onnx  = "resnet50_trt_fp32_tmp.onnx"
    trt_tmp    = "resnet50_int8_tmp.trt"
    calib_cache = "trt_calib_tmp.cache"

    try:
        export_to_onnx(fp32_model, fp32_onnx)

        # ── Calibrator ────────────────────────────────────────────────────────
        class CIFARCalibrator(trt.IInt8EntropyCalibrator2):
            def __init__(self, loader, cache_file: str):
                super().__init__()
                self._loader     = iter(loader)
                self._batches    = 0
                self._cache_file = cache_file
                self._device_mem = None
                first_batch, _ = next(iter(loader))
                self._batch_size = first_batch.shape[0]
                self._device_mem = cuda.mem_alloc(
                    int(np.prod(first_batch.shape) * 4)
                )

            def get_batch_size(self):
                return self._batch_size

            def get_batch(self, names):
                if self._batches >= CALIB_BATCHES:
                    return None
                try:
                    imgs, _ = next(self._loader)
                    cuda.memcpy_htod(self._device_mem,
                                     imgs.numpy().astype(np.float32))
                    self._batches += 1
                    return [int(self._device_mem)]
                except StopIteration:
                    return None

            def read_calibration_cache(self):
                if os.path.exists(self._cache_file):
                    with open(self._cache_file, "rb") as f:
                        return f.read()
                return None

            def write_calibration_cache(self, cache):
                with open(self._cache_file, "wb") as f:
                    f.write(cache)

        # ── Build engine ──────────────────────────────────────────────────────
        builder = trt.Builder(TRT_LOGGER)
        network = builder.create_network(
            1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)
        )
        parser = trt.OnnxParser(network, TRT_LOGGER)
        with open(fp32_onnx, "rb") as f:
            if not parser.parse(f.read()):
                raise RuntimeError(f"ONNX parse error: {parser.get_error(0)}")

        config = builder.create_builder_config()
        config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)
        config.set_flag(trt.BuilderFlag.INT8)
        config.set_flag(trt.BuilderFlag.FP16)
        config.int8_calibrator = CIFARCalibrator(calib_loader, calib_cache)

        profile = builder.create_optimization_profile()
        profile.set_shape("input",
                          min=(1, 3, 32, 32),
                          opt=(BATCH_SIZE, 3, 32, 32),
                          max=(BATCH_SIZE * 2, 3, 32, 32))
        config.add_optimization_profile(profile)

        print("        Building TRT INT8 engine (may take a few minutes)…")
        t0         = time.perf_counter()
        serialized = builder.build_serialized_network(network, config)
        build_sec  = time.perf_counter() - t0
        print(f"        Engine built in {build_sec:.1f}s")

        with open(trt_tmp, "wb") as f:
            f.write(serialized)

        # ── Deserialise and create execution context ──────────────────────────
        runtime = trt.Runtime(TRT_LOGGER)
        engine  = runtime.deserialize_cuda_engine(serialized)
        context = engine.create_execution_context()
        context.set_input_shape("input", (BATCH_SIZE, 3, 32, 32))

        d_input  = cuda.mem_alloc(BATCH_SIZE * 3 * 32 * 32 * 4)
        d_output = cuda.mem_alloc(BATCH_SIZE * NUM_CLASSES * 4)
        stream   = cuda.Stream()

        def infer_trt_batch(x_np: np.ndarray) -> np.ndarray:
            cuda.memcpy_htod_async(d_input, x_np.astype(np.float32), stream)
            context.execute_async_v2(
                bindings     = [int(d_input), int(d_output)],
                stream_handle= stream.handle,
            )
            out = np.empty((BATCH_SIZE, NUM_CLASSES), dtype=np.float32)
            cuda.memcpy_dtoh_async(out, d_output, stream)
            stream.synchronize()
            return out

        # Single-image context (separate binding shapes)
        ctx1    = engine.create_execution_context()
        ctx1.set_input_shape("input", (1, 3, 32, 32))
        d_in1   = cuda.mem_alloc(1 * 3 * 32 * 32 * 4)
        d_out1  = cuda.mem_alloc(1 * NUM_CLASSES * 4)

        def infer_trt_single(x_np: np.ndarray) -> np.ndarray:
            cuda.memcpy_htod_async(d_in1, x_np.astype(np.float32), stream)
            ctx1.execute_async_v2(
                bindings     = [int(d_in1), int(d_out1)],
                stream_handle= stream.handle,
            )
            out = np.empty((1, NUM_CLASSES), dtype=np.float32)
            cuda.memcpy_dtoh_async(out, d_out1, stream)
            stream.synchronize()
            return out

        # ── Accuracy evaluation ───────────────────────────────────────────────
        from sklearn.metrics import (accuracy_score, precision_score,
                                     recall_score, f1_score)
        preds, labels = [], []
        for inputs, lbls in test_loader:
            out = infer_trt_batch(inputs.numpy())
            preds.extend(np.argmax(out, axis=1))
            labels.extend(lbls.numpy())
        y_pred, y_true = np.array(preds), np.array(labels)
        metrics = {
            "accuracy" : float(accuracy_score(y_true, y_pred)),
            "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
            "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
            "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        }

        # ── Latency (CUDA events via pycuda) ──────────────────────────────────
        dummy_b  = np.random.randn(BATCH_SIZE, 3, 32, 32).astype(np.float32)
        dummy_s  = np.random.randn(1,          3, 32, 32).astype(np.float32)
        start_ev = cuda.Event()
        end_ev   = cuda.Event()

        for _ in range(5):
            infer_trt_batch(dummy_b)
            infer_trt_single(dummy_s)

        batch_times, single_times = [], []
        for _ in range(INFERENCE_RUNS):
            start_ev.record()
            infer_trt_batch(dummy_b)
            end_ev.record()
            end_ev.synchronize()
            batch_times.append(start_ev.time_till(end_ev))

            start_ev.record()
            infer_trt_single(dummy_s)
            end_ev.record()
            end_ev.synchronize()
            single_times.append(start_ev.time_till(end_ev))

        batch_ms  = float(np.mean(batch_times))
        single_ms = float(np.mean(single_times))
        infer_ms  = {
            "single_img_gpu_ms"  : round(single_ms, 4),
            "batch128_gpu_ms"    : round(batch_ms, 4),
            "per_img_gpu_ms"     : round(batch_ms / BATCH_SIZE, 4),
            "throughput_imgs_sec": round(BATCH_SIZE / (batch_ms / 1000), 1),
            "timing_method"      : "CUDA events + torch.cuda.synchronize()",
        }

        size_mb  = disk_size_mb(trt_tmp)
        result   = build_entry(
            metrics      = metrics,
            params       = baseline_params,
            size_mb      = size_mb,
            flops_G      = flops_G,
            inference_ms = infer_ms,
        )

        # Persist
        import shutil
        save_path = os.path.join(SAVE_DIR, "tensorrt_int8.trt")
        shutil.copy(trt_tmp, save_path)

        print(f"        → Acc: {metrics['accuracy']:.4f}  "
              f"Disk: {size_mb:.2f} MB  "
              f"Lat: {batch_ms:.1f} ms")
        print(f"        ✓ Saved → {save_path}")
        return result

    except Exception as exc:
        print(f"        → FAILED: {exc}")
        return None
    finally:
        for f in [fp32_onnx, trt_tmp, calib_cache]:
            if os.path.exists(f):
                os.remove(f)


# ══════════════════════════════════════════════════════════════════════════════
# 3a. PyTorch Dynamic PTQ on CUDA
# ══════════════════════════════════════════════════════════════════════════════
def run_dynamic_ptq(fp32_model, test_loader, flops_G: float, device) -> Optional[Dict]:
    """
    Dynamic PTQ: only Linear layer weights are stored as INT8.

    On CUDA the weights are dequantised to FP16/FP32 before the cuBLAS GEMM
    — this is NOT true INT8 GEMM.  The benefit is reduced memory bandwidth
    (~2× for Linear layers); Conv2d stays FP32.
    No calibration data required.
    """
    import torch.nn as nn
    print("\n        3a. Dynamic PTQ (Linear → qint8) on CUDA")
    os.makedirs(SAVE_DIR, exist_ok=True)

    try:
        q_dyn = torch.quantization.quantize_dynamic(
            copy.deepcopy(fp32_model), {nn.Linear}, dtype=torch.qint8
        )
        q_dyn.eval()

        # Save (CPU state dict — CUDA dynamic quant isn't directly serialisable
        # with torch.save on the GPU model)
        save_path = os.path.join(SAVE_DIR, "dynamic_ptq_int8.pth")
        torch.save(q_dyn.cpu().state_dict(), save_path)
        print(f"        ✓ Saved → {save_path}")

        q_dyn = q_dyn.to(device)
        metrics  = evaluate_torch(q_dyn, test_loader, device)
        infer_ms = measure_gpu_throughput(q_dyn, device)
        params   = count_params(q_dyn.cpu())
        size_mb  = disk_size_mb(q_dyn.cpu())

        result = build_entry(
            metrics      = metrics,
            params       = params,
            size_mb      = size_mb,
            flops_G      = flops_G,
            inference_ms = infer_ms,
        )
        print(f"           → Acc: {metrics['accuracy']:.4f}  "
              f"GPU: {infer_ms['batch128_gpu_ms']:.1f} ms  "
              f"Disk: {size_mb:.2f} MB")
        return result

    except Exception as exc:
        print(f"           → FAILED: {exc}")
        return None


# ══════════════════════════════════════════════════════════════════════════════
# 3b. PT2E + X86InductorQuantizer → torch.compile
# ══════════════════════════════════════════════════════════════════════════════
def _resolve_pt2e_imports():
    """
    Resolve PT2E symbols across PyTorch 2.3 / 2.4 / 2.5+ import paths.
    Returns (prepare_pt2e, convert_pt2e, Quantizer, get_config, export_fn).
    """
    errors = []

    prepare_pt2e = convert_pt2e = None
    for mod_path in [
        "torch.ao.quantization.quantize_pt2e",
        "torch.quantization.quantize_pt2e",
    ]:
        try:
            import importlib
            m = importlib.import_module(mod_path)
            prepare_pt2e = m.prepare_pt2e
            convert_pt2e = m.convert_pt2e
            break
        except (ImportError, AttributeError) as exc:
            errors.append(f"  {mod_path}: {exc}")

    if prepare_pt2e is None:
        raise ImportError("Cannot find prepare_pt2e / convert_pt2e.\n"
                          + "\n".join(errors) + "\nRequires PyTorch >= 2.3.")

    Quantizer = get_config = None
    for mod_path, cls, cfg in [(
        "torch.ao.quantization.quantizer.x86_inductor_quantizer",
        "X86InductorQuantizer",
        "get_default_x86_inductor_quantization_config",
    )]:
        try:
            import importlib
            m = importlib.import_module(mod_path)
            Quantizer  = getattr(m, cls)
            get_config = getattr(m, cfg)
            break
        except (ImportError, AttributeError) as exc:
            errors.append(f"  {mod_path}.{cls}: {exc}")

    if Quantizer is None:
        raise ImportError("Cannot find X86InductorQuantizer.\n" + "\n".join(errors))

    export_fn = None
    try:
        from torch.export import export as _export
        def export_fn(model, example_args):
            return _export(model, example_args).module()
    except (ImportError, AttributeError) as exc:
        errors.append(f"  torch.export.export: {exc}")

    if export_fn is None:
        try:
            from torch._export import capture_pre_autograd_graph as _cap
            def export_fn(model, example_args):
                return _cap(model, example_args)
        except (ImportError, AttributeError) as exc:
            errors.append(f"  torch._export.capture_pre_autograd_graph: {exc}")

    if export_fn is None:
        raise ImportError("Cannot find graph export function.\n" + "\n".join(errors))

    return prepare_pt2e, convert_pt2e, Quantizer, get_config, export_fn


def run_pt2e_ptq(fp32_model, test_loader, calib_loader,
                 flops_G: float, device) -> Optional[Dict]:
    """
    PT2E (PyTorch 2 Export) + X86InductorQuantizer + torch.compile.

    Pipeline:
      torch.export → prepare_pt2e (insert MinMaxObservers)
      → calibrate on CALIB_BATCHES → convert_pt2e (freeze S/Z)
      → torch.compile(inductor, mode='max-autotune')

    Quantisation math (MinMaxObserver):
      S = (x_max − x_min) / (2^8 − 1)
      Z = clip(round(−x_min / S), 0, 255)
      Q(x) = clip(round(x / S) + Z, 0, 255)  → UINT8 activations

    Weights: INT8 per-channel symmetric
    Activations: UINT8 per-tensor affine

    NOTE: PT2E on CUDA (torch 2.3–2.5) is still maturing;
          some ops may fall back to FP16 at compile time.
    """
    print("\n        3b. PT2E + X86InductorQuantizer → torch.compile")
    os.makedirs(SAVE_DIR, exist_ok=True)

    try:
        (prepare_pt2e, convert_pt2e,
         Quantizer, get_config, export_fn) = _resolve_pt2e_imports()
        print(f"           PT2E imports resolved (torch {torch.__version__})")

        example_args = (torch.randn(1, 3, 32, 32, device=device),)
        m = copy.deepcopy(fp32_model).to(device).eval()
        exported = export_fn(m, example_args)

        quantizer = Quantizer()
        quantizer.set_global(get_config())
        prepared = prepare_pt2e(exported, quantizer)

        prepared.eval()
        with torch.no_grad():
            for i, (imgs, _) in enumerate(calib_loader):
                prepared(imgs.to(device))
                if i + 1 >= CALIB_BATCHES:
                    break

        q_pt2e = convert_pt2e(prepared)

        # Save pre-compile state dict (compiled graph not trivially serialisable)
        save_path = os.path.join(SAVE_DIR, "pt2e_int8.pth")
        try:
            torch.save(q_pt2e.state_dict(), save_path)
            print(f"        ✓ Saved → {save_path}")
        except Exception as exc:
            print(f"        ⚠ PT2E state-dict save skipped: {exc}")

        q_compiled = torch.compile(q_pt2e, mode="max-autotune", backend="inductor")

        metrics  = evaluate_torch(q_compiled, test_loader, device)
        infer_ms = measure_gpu_throughput(q_compiled, device)
        params   = count_params(q_pt2e)          # use pre-compile model

        # Compiled model can't be trivially serialised → size_mb = None
        result = build_entry(
            metrics      = metrics,
            params       = params,
            size_mb      = None,
            flops_G      = flops_G,
            inference_ms = infer_ms,
        )
        print(f"           → Acc: {metrics['accuracy']:.4f}  "
              f"GPU: {infer_ms['batch128_gpu_ms']:.1f} ms")
        return result

    except ImportError as exc:
        print(f"           → SKIPPED (PT2E unavailable): {exc}")
        return None
    except Exception as exc:
        print(f"           → FAILED: {exc}")
        return None


# ══════════════════════════════════════════════════════════════════════════════
# FP32 Baseline
# ══════════════════════════════════════════════════════════════════════════════
def benchmark_fp32(fp32_model, test_loader, flops_G: float, device) -> Dict:
    print(f"\n  [0/3] FP32 GPU Baseline")
    model   = copy.deepcopy(fp32_model).to(device).eval()
    metrics = evaluate_torch(model, test_loader, device)
    infer_ms = measure_gpu_throughput(model, device)
    params  = count_params(model.cpu())
    size_mb = disk_size_mb(model.cpu())
    result  = build_entry(
        metrics      = metrics,
        params       = params,
        size_mb      = size_mb,
        flops_G      = flops_G,
        inference_ms = infer_ms,
    )
    print(f"        Acc: {metrics['accuracy']:.4f}  "
          f"GPU: {infer_ms['batch128_gpu_ms']:.1f} ms  "
          f"Disk: {size_mb:.2f} MB")
    return result


# ══════════════════════════════════════════════════════════════════════════════
# Main
# ══════════════════════════════════════════════════════════════════════════════
def main():
    parser = argparse.ArgumentParser(description="GPU PTQ — ResNet-50 / CIFAR-10")
    parser.add_argument("--install-deps", action="store_true",
                        help="Install all GPU PTQ dependencies and exit")
    parser.add_argument("--device", default="cuda",
                        help="Compute device: cuda | cuda:0 | cpu")
    args, _ = parser.parse_known_args()

    if args.install_deps:
        install_deps()
        return

    # ── Dependency check ───────────────────────────────────────────────────────
    missing = []
    if torch       is None: missing.append("torch          → pip install torch --index-url https://download.pytorch.org/whl/cu121")
    if onnx        is None: missing.append("onnx           → pip install onnx")
    if ort         is None: missing.append("onnxruntime-gpu → pip install onnxruntime-gpu")
    if missing:
        print("\n⚠  Missing required packages:")
        for m in missing:
            print(f"   {m}")
        print("\nRun:  python gpu_ptq.py --install-deps")
        sys.exit(1)

    device = torch.device(args.device if torch.cuda.is_available() else "cpu")

    print(f"\n{'='*65}")
    print("  GPU PTQ — ResNet-50 / CIFAR-10")
    print(f"  Device : {device}")
    if torch.cuda.is_available():
        print(f"  GPU    : {torch.cuda.get_device_name(device)}")
        print(f"  CUDA   : {torch.version.cuda}")
        print(f"  cuDNN  : {torch.backends.cudnn.version()}")
    print("  Methods: ORT (MinMax|Entropy|Percentile) | TensorRT | Dynamic | PT2E")
    print(f"{'='*65}")

    # ── Load baseline ──────────────────────────────────────────────────────────
    if not os.path.exists(BASELINE_MODEL_PATH):
        print(f"\n✗ Baseline model not found: {BASELINE_MODEL_PATH}")
        print("  Run the baseline training script first.")
        sys.exit(1)

    fp32_model   = load_baseline(BASELINE_MODEL_PATH)
    test_loader  = get_test_loader()
    calib_loader = get_calib_loader()

    # ── FLOPs (identical for FP32 and INT8 — computed once) ───────────────────
    print("\n  Computing FLOPs (op count — identical for FP32 and INT8)…")
    flops_info   = count_flops(fp32_model)
    flops_G      = flops_info["flops_G"]
    params_total = flops_info["params_total"]
    print(f"  FLOPs : {flops_G:.4f} GFLOPs  "
          f"Params: {flops_info['params_M']:.2f}M  "
          f"(tool: {flops_info['tool']})")
    print("  Note  : FLOPs unchanged by quantisation. "
          "INT8 speedup = hardware SIMD, not fewer ops.")

    # Baseline param dict (used for ONNX/TRT entries which share the same arch)
    baseline_params = count_params(fp32_model)

    # ── Run all methods ────────────────────────────────────────────────────────
    output: Dict = {}

    # 0. FP32 baseline
    output["fp32_baseline"] = benchmark_fp32(
        fp32_model, test_loader, flops_G, device
    )

    # 1. ONNX Runtime (3 calibration methods)
    ort_results = run_onnx_ptq(
        fp32_model, test_loader, calib_loader, flops_G, baseline_params
    )
    for key, val in ort_results.items():
        if val is not None:
            output[key] = val

    # 2. TensorRT
    trt_result = run_tensorrt_ptq(
        fp32_model, test_loader, calib_loader, flops_G, baseline_params
    )
    if trt_result is not None:
        output["tensorrt_int8"] = trt_result

    # 3a. PyTorch Dynamic PTQ
    print("\n  [3/3] PyTorch CUDA — Dynamic PTQ + PT2E Inductor INT8")
    dyn_result = run_dynamic_ptq(fp32_model, test_loader, flops_G, device)
    if dyn_result is not None:
        output["dynamic_ptq"] = dyn_result

    # 3b. PT2E Inductor
    pt2e_result = run_pt2e_ptq(
        fp32_model, test_loader, calib_loader, flops_G, device
    )
    if pt2e_result is not None:
        output["pt2e_inductor"] = pt2e_result

    # ── Persist JSON ───────────────────────────────────────────────────────────
    with open(OUTPUT_JSON, "w") as f:
        json.dump(output, f, indent=2)

    # ── Console summary table ──────────────────────────────────────────────────
    baseline_acc = output["fp32_baseline"]["accuracy"]

    print(f"\n{'='*80}")
    print(f"  ✓ Results saved → {OUTPUT_JSON}")
    print(f"  ✓ Models saved  → {SAVE_DIR}/")
    print(f"\n  {'Method':<22} {'Acc':>7} {'Drop':>7} {'F1':>7} "
          f"{'Size(MB)':>9} {'Batch(ms)':>10} {'Imgs/s':>8} {'GFLOPs':>8}")
    print(f"  {'-'*80}")

    for key, val in output.items():
        if val is None:
            continue
        drop     = baseline_acc - val["accuracy"] if key != "fp32_baseline" else 0.0
        size_str = f"{val['size_mb']:.2f}" if val.get("size_mb") is not None else "  N/A"
        lat      = val["inference_ms"]["batch128_gpu_ms"]
        tput     = val["inference_ms"]["throughput_imgs_sec"]
        gf       = val["flops_G"]
        print(f"  {key:<22} {val['accuracy']:.4f}  {drop:+.4f}  "
              f"{val['f1']:.4f}  {size_str:>9}  {lat:>10.1f}  "
              f"{tput:>8.1f}  {gf:>8.4f}")

    # Best quantised method (lowest accuracy drop, then smallest size)
    quant_entries = {k: v for k, v in output.items()
                     if v is not None and k != "fp32_baseline"}
    if quant_entries:
        best_key = min(
            quant_entries,
            key=lambda k: (
                baseline_acc - quant_entries[k]["accuracy"],
                quant_entries[k]["size_mb"] or 999,
            ),
        )
        best = quant_entries[best_key]
        fp32_lat  = output["fp32_baseline"]["inference_ms"]["batch128_gpu_ms"]
        best_lat  = best["inference_ms"]["batch128_gpu_ms"]
        speedup   = fp32_lat / best_lat if best_lat else 0.0
        print(f"\n  Best config : {best_key}")
        print(f"  Accuracy    : {best['accuracy']:.4f}  "
              f"(drop: {baseline_acc - best['accuracy']:+.4f})")
        print(f"  Speedup     : {speedup:.2f}× vs FP32  "
              f"({best_lat:.1f} ms vs {fp32_lat:.1f} ms batch-128)")
    print(f"{'='*80}\n")


if __name__ == "__main__":
    main()


  GPU PTQ — ResNet-50 / CIFAR-10
  Device : cuda
  GPU    : NVIDIA GeForce RTX 5070 Laptop GPU
  CUDA   : 12.8
  cuDNN  : 92000
  Methods: ORT (MinMax|Entropy|Percentile) | TensorRT | Dynamic | PT2E

  Computing FLOPs (op count — identical for FP32 and INT8)…
  FLOPs : 2.5957 GFLOPs  Params: 23.52M  (tool: manual)
  Note  : FLOPs unchanged by quantisation. INT8 speedup = hardware SIMD, not fewer ops.

  [0/3] FP32 GPU Baseline
        Acc: 0.9320  GPU: 51.1 ms  Disk: 90.05 MB

  [1/3] ONNX Runtime — Static INT8 PTQ (QDQ)


W0505 15:27:07.094000 28988 Lib\site-packages\torch\onnx\_internal\exporter\_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0505 15:27:07.733000 28988 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


...

 → Acc: 0.9319  Disk: 90.18 MB  Lat: 64.0 ms  (CUDAExecutionProvider)
        ✓ Saved → __1__saved_models_PTQ_GPU\onnx_int8_minmax.onnx
        Calibration: Entropy     

Finding optimal threshold for each tensor using 'entropy' algorithm ...
Number of tensors : 122
Number of histogram bins : 128 (The number may increase depends on the data it collects)
Number of quantized bins : 128


 → Acc: 0.9319  Disk: 90.18 MB  Lat: 63.9 ms  (CUDAExecutionProvider)
        ✓ Saved → __1__saved_models_PTQ_GPU\onnx_int8_entropy.onnx
        Calibration: Percentile  

Finding optimal threshold for each tensor using 'percentile' algorithm ...
Number of tensors : 122
Number of histogram bins : 2048
Percentile : (0.0010000000000047748,99.999)


 → Acc: 0.9316  Disk: 90.18 MB  Lat: 63.9 ms  (CUDAExecutionProvider)
        ✓ Saved → __1__saved_models_PTQ_GPU\onnx_int8_percentile.onnx

  [2/3] TensorRT — INT8 PTQ
        ⚠ TensorRT/pycuda not available: No module named 'tensorrt'
          Install: pip install tensorrt pycuda

  [3/3] PyTorch CUDA — Dynamic PTQ + PT2E Inductor INT8

        3a. Dynamic PTQ (Linear → qint8) on CUDA
        ✓ Saved → __1__saved_models_PTQ_GPU\dynamic_ptq_int8.pth
           → FAILED: Could not run 'quantized::linear_dynamic' with arguments from the 'CUDA' backend. This could be because the operator doesn't exist for this backend, or was omitted during the selective/custom build process (if using custom build). If you are a Facebook employee using PyTorch on mobile, please visit https://fburl.com/ptmfixes for possible resolutions. 'quantized::linear_dynamic' is only available for these backends: [CPU, Meta, BackendSelect, Python, FuncTorchDynamicLayerBackMode, Functionalize, Named, Conjugate, Negat